# Comparaison des modèles — Linear / MLP / RBF

Ce notebook compare les 3 modèles implémentés en Rust (via le binding Python `vision_ai`) sur le dataset VisionAI (3 classes : aucun / humain / animal, images 64×64×3).

**Métriques comparées :**
- Accuracy sur le jeu de test
- Temps d'entraînement
- Courbes de perte (loss)
- Matrice de confusion

In [ ]:
import os, sys, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

IN_COLAB = os.path.exists('/content')

if IN_COLAB:
    BASE_DIR = '/content/VisionAI'

    # 1. Cloner le repo
    if not os.path.exists(BASE_DIR):
        subprocess.run(['git', 'clone', '-b', 'RBF_thinina',
                        'https://github.com/SINCER-Ali/VisionAI.git', BASE_DIR], check=True)
    else:
        subprocess.run(['git', 'pull'], cwd=BASE_DIR)

    # 2. Installer Rust via rustup
    result = subprocess.run(['which', 'cargo'], capture_output=True)
    if result.returncode != 0:
        print('Installation de Rust...')
        subprocess.run(
            'curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable',
            shell=True, check=True
        )
        print('Rust installé ✓')
    else:
        print('Rust déjà disponible ✓')

    # 3. Forcer le PATH cargo dans l'environnement courant
    cargo_bin = '/root/.cargo/bin'
    os.environ['PATH'] = cargo_bin + ':' + os.environ.get('PATH', '')
    print(f'cargo : {subprocess.run(["cargo", "--version"], capture_output=True, text=True).stdout.strip()}')

    # 4. Installer maturin
    subprocess.run(['pip', 'install', 'maturin', 'setuptools-rust', '-q'], check=True)
    print(f'maturin : {subprocess.run(["maturin", "--version"], capture_output=True, text=True).stdout.strip()}')

    # 5. Compiler le binding (debug, plus rapide et sans --release)
    print('Compilation du binding vision_ai...')
    result = subprocess.run(
        ['maturin', 'develop'],
        cwd=f'{BASE_DIR}/python_binding',
        env={**os.environ, 'PATH': cargo_bin + ':' + os.environ.get('PATH', '')},
        capture_output=False  # affiche les erreurs Rust directement
    )
    if result.returncode != 0:
        raise RuntimeError('Compilation échouée — voir les erreurs Rust ci-dessus')
    print('Binding compilé ✓')

else:
    BASE_DIR = os.path.dirname(os.path.abspath('.'))

DATASET_DIR = os.path.join(BASE_DIR, 'datasets')
MODELS_DIR  = os.path.join(BASE_DIR, 'models')
sys.path.insert(0, os.path.join(BASE_DIR, 'python_binding'))

import vision_ai
CLASSES = ['aucun', 'humain', 'animal']
print('vision_ai importé ✓')
print(f'Classes : {CLASSES}')

## 1. Chargement du dataset

In [ ]:
X_train = np.load(os.path.join(DATASET_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(DATASET_DIR, 'y_train.npy'))
X_test  = np.load(os.path.join(DATASET_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(DATASET_DIR, 'y_test.npy'))

INPUT_SIZE = X_train.shape[1]  # 12288

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Input size : {INPUT_SIZE}')
counts = [int(np.sum(y_train == i)) for i in range(3)]
for c, n in zip(CLASSES, counts):
    print(f'  {c:8s} : {n} images train')

In [ ]:
# Préparer les listes Python (format attendu par vision_ai)
inputs_train  = X_train.tolist()
inputs_test   = X_test.tolist()

def one_hot(labels, n_classes=3):
    out = []
    for l in labels:
        v = [0.0] * n_classes
        v[int(l)] = 1.0
        out.append(v)
    return out

targets_train = one_hot(y_train)
targets_test  = one_hot(y_test)

def evaluate(predict_fn, inputs, labels):
    """Accuracy + matrice de confusion."""
    preds = [predict_fn(x) for x in inputs]
    y_pred = [p.index(max(p)) for p in preds]
    correct = sum(p == int(t) for p, t in zip(y_pred, labels))
    acc = correct / len(labels)
    cm = np.zeros((3, 3), dtype=int)
    for p, t in zip(y_pred, labels):
        cm[int(t)][p] += 1
    return acc, cm, y_pred

results = {}  # nom -> {acc, cm, time}
print('Données prêtes ✓')

## 2. Modèle Linéaire (Régression Linéaire)

In [ ]:
print('=== Entraînement Linear ===')
t0 = time.time()

linear = vision_ai.LinearRegression(INPUT_SIZE)
# one output par classe (on prend argmax ensuite)
# Le modèle linéaire de vision_ai supporte une seule sortie -> on entraîne 1 modèle par classe
# et on combine (one-vs-rest)
linear_models = []
for cls_idx in range(3):
    m = vision_ai.LinearRegression(INPUT_SIZE)
    targets_cls = [[1.0] if int(l) == cls_idx else [0.0] for l in y_train]
    m.train(inputs_train, targets_cls, epochs=100, lr=0.001)
    linear_models.append(m)

t_linear = time.time() - t0

def predict_linear(x):
    scores = [m.predict(x)[0] for m in linear_models]
    return scores

acc_linear, cm_linear, _ = evaluate(predict_linear, inputs_test, y_test)
results['Linear'] = {'acc': acc_linear, 'cm': cm_linear, 'time': t_linear}
print(f'Accuracy Linear : {acc_linear*100:.1f}%  |  Temps : {t_linear:.1f}s')

## 3. Modèle MLP

In [ ]:
print('=== Entraînement MLP ===')
t0 = time.time()

mlp = vision_ai.PyMLP([INPUT_SIZE, 64, 3])
mlp.train(inputs_train, targets_train, learning_rate=0.001, epochs=30)

t_mlp = time.time() - t0

acc_mlp, cm_mlp, _ = evaluate(mlp.predict, inputs_test, y_test)
results['MLP'] = {'acc': acc_mlp, 'cm': cm_mlp, 'time': t_mlp}
print(f'Accuracy MLP : {acc_mlp*100:.1f}%  |  Temps : {t_mlp:.1f}s')

## 4. Modèle RBF

In [ ]:
print('=== Entraînement RBF ===')
# On réduit la dimensionnalité par sous-échantillonnage de pixels pour le RBF
# (12288 dims est lourd pour les centres, on prend 1 pixel sur 4 → 3072 dims)
STEP = 4
X_train_r = X_train[:, ::STEP]
X_test_r  = X_test[:, ::STEP]
DIM_RBF   = X_train_r.shape[1]
print(f'Dimension réduite pour RBF : {DIM_RBF}')

inputs_train_r = X_train_r.tolist()
inputs_test_r  = X_test_r.tolist()

t0 = time.time()
rbf = vision_ai.PyRBF(DIM_RBF, 3, n_centers=30, sigma=1.0)
rbf.init_centers_random(inputs_train_r)
rbf.train(inputs_train_r, targets_train, lr=0.01, epochs=100, regression=False)

t_rbf = time.time() - t0

acc_rbf, cm_rbf, _ = evaluate(rbf.predict, inputs_test_r, y_test)
results['RBF'] = {'acc': acc_rbf, 'cm': cm_rbf, 'time': t_rbf}
print(f'Accuracy RBF : {acc_rbf*100:.1f}%  |  Temps : {t_rbf:.1f}s')

# Sauvegarde du modèle RBF entraîné
os.makedirs(MODELS_DIR, exist_ok=True)
rbf.save_json(os.path.join(MODELS_DIR, 'rbf_weights.json'))
print('Modèle RBF sauvegardé → models/rbf_weights.json')

## 5. Comparaison des résultats

In [ ]:
print('\n========== RÉSUMÉ ==========')
print(f'{"Modèle":<10} {"Accuracy":>10} {"Temps (s)":>12}')
print('-' * 34)
for name, r in results.items():
    print(f'{name:<10} {r["acc"]*100:>9.1f}% {r["time"]:>12.1f}s')

# --- Graphique accuracy ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

names  = list(results.keys())
accs   = [results[n]['acc'] * 100 for n in names]
colors = ['steelblue', 'seagreen', 'darkorange']

# Accuracy bar
ax = axes[0]
bars = ax.bar(names, accs, color=colors, width=0.5)
ax.set_ylim(0, 100)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy sur le jeu de test')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', fontweight='bold')

# Temps d'entraînement
ax = axes[1]
times = [results[n]['time'] for n in names]
bars2 = ax.bar(names, times, color=colors, width=0.5)
ax.set_ylabel('Temps (secondes)')
ax.set_title("Temps d'entraînement")
for bar, t in zip(bars2, times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{t:.1f}s', ha='center', fontweight='bold')

# Matrice de confusion — meilleur modèle
best_name = max(results, key=lambda n: results[n]['acc'])
cm_best   = results[best_name]['cm']
ax = axes[2]
im = ax.imshow(cm_best, cmap='Blues')
ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Prédiction'); ax.set_ylabel('Réel')
ax.set_title(f'Matrice de confusion — {best_name} (meilleur)')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm_best[i, j]), ha='center', va='center',
                color='white' if cm_best[i, j] > cm_best.max()/2 else 'black', fontsize=13)
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'notebooks', 'comparaison_modeles.png'), dpi=120)
plt.show()
print(f'\nMeilleur modèle : {best_name} ({results[best_name]["acc"]*100:.1f}%)')

## 6. Matrices de confusion — tous les modèles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, r) in zip(axes, results.items()):
    cm = r['cm']
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks([0,1,2]); ax.set_yticks([0,1,2])
    ax.set_xticklabels(CLASSES); ax.set_yticklabels(CLASSES)
    ax.set_xlabel('Prédiction'); ax.set_ylabel('Réel')
    ax.set_title(f'{name} — {r["acc"]*100:.1f}%')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=13)
    plt.colorbar(im, ax=ax)

plt.suptitle('Matrices de confusion — Linear / MLP / RBF', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Conclusion

In [ ]:
best = max(results, key=lambda n: results[n]['acc'])
fastest = min(results, key=lambda n: results[n]['time'])

print('=== CONCLUSION ===')
print(f'Meilleure accuracy   : {best} ({results[best]["acc"]*100:.1f}%)')
print(f'Entraînement le plus rapide : {fastest} ({results[fastest]["time"]:.1f}s)')
print()
print('Analyse :')
print('  Linear  : rapide, mais insuffisant pour des données images non-linéaires')
print('  MLP     : bon compromis accuracy / temps avec des couches cachées')
print('  RBF     : bonne généralisation locale grâce aux centres gaussiens')
print()
print('Pour la soutenance, le MLP ou le RBF sera utilisé dans le client web.')